# Boundary-aware attention results

This lightweight notebook reads `boundary_attention_artifacts.zip`, compares the
controlled runs, and shows boundary diagnostics and paired examples. It performs no
training, checkpoint loading, inference, or test-set access.

## 0. Load the downloaded artifact

In [1]:
import csv, io, json, statistics, sys, zipfile
from pathlib import Path
from IPython.display import display, Markdown

NAME = "boundary_attention_artifacts.zip"
candidates = [Path(NAME), Path("/content") / NAME,
              Path("/content/drive/MyDrive/CITS4012_A2_boundary_attention") / NAME]
archive = next((p for p in candidates if p.exists()), None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    archive = next((p for p in candidates if p.exists()), None)
if archive is None and "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload(); assert NAME in uploaded
    archive = Path("/content") / NAME; archive.write_bytes(uploaded[NAME])
assert archive is not None, f"Place {NAME} beside this notebook or upload it."
zf = zipfile.ZipFile(archive)
required = {"boundary_results/runs.csv", "boundary_results/boundary_summary.json",
            "boundary_results/boundary_details.csv"}
assert required <= set(zf.namelist()), f"Missing: {sorted(required-set(zf.namelist()))}"
runs = list(csv.DictReader(io.StringIO(zf.read("boundary_results/runs.csv").decode())))
summary = json.loads(zf.read("boundary_results/boundary_summary.json"))
details = list(csv.DictReader(io.StringIO(zf.read("boundary_results/boundary_details.csv").decode())))
print("Loaded", archive, "| rows", len(details), "| no training or inference")


Loaded boundary_attention_artifacts.zip | rows 6456 | no training or inference


In [2]:
models = [("BoundaryControl_v2", "Current v2"),
          ("BoundaryAware_start2end", "Start-conditioned end head")]
metrics = ("bleu1", "meteor", "rougeL", "em", "f1")
lines = ["| Model | BLEU-1 | METEOR | ROUGE-L | EM | F1 |", "|---|---:|---:|---:|---:|---:|"]
for key, label in models:
    cells = [f"{summary['models'][key]['metrics'][m][0]:.2f} ± {summary['models'][key]['metrics'][m][1]:.2f}" for m in metrics]
    lines.append("| " + label + " | " + " | ".join(cells) + " |")
display(Markdown("## 1. Validation results over three seeds\n\n" + "\n".join(lines)))
deltas = {m: statistics.mean([v[m] for v in summary["paired_seed_deltas"].values()]) for m in metrics}
display(Markdown(f"Paired mean change (new − v2): **ROUGE-L {deltas['rougeL']:+.2f}**, "
                 f"**F1 {deltas['f1']:+.2f}**, **EM {deltas['em']:+.2f}** points."))


## 1. Validation results over three seeds

| Model | BLEU-1 | METEOR | ROUGE-L | EM | F1 |
|---|---:|---:|---:|---:|---:|
| Current v2 | 38.89 ± 0.65 | 33.52 ± 0.78 | 44.27 ± 0.80 | 27.26 ± 0.39 | 44.37 ± 0.74 |
| Start-conditioned end head | 38.25 ± 0.37 | 33.23 ± 0.61 | 43.70 ± 0.49 | 26.70 ± 0.35 | 43.77 ± 0.56 |

Paired mean change (new − v2): **ROUGE-L -0.57**, **F1 -0.60**, **EM -0.56** points.

In [3]:
lines = ["| Model | Start exact | End exact | Full span exact | Span IoU |",
         "|---|---:|---:|---:|---:|"]
for key, label in models:
    b = summary["models"][key]["boundary"]
    vals = [b[x] for x in ("start_exact", "end_exact", "span_exact")]
    vals.append(summary["models"][key]["span_iou"])
    lines.append("| " + label + " | " + " | ".join(f"{m:.2f} ± {s:.2f}" for m, s in vals) + " |")
display(Markdown("## 2. Boundary diagnostics on trainable validation examples\n\n" + "\n".join(lines)))
display(Markdown("Start/end exactness is measured against the deterministic best silver span. "
                 "It diagnoses boundary selection; it is not a substitute for answer-text metrics."))


## 2. Boundary diagnostics on trainable validation examples

| Model | Start exact | End exact | Full span exact | Span IoU |
|---|---:|---:|---:|---:|
| Current v2 | 50.58 ± 1.34 | 50.47 ± 1.13 | 37.40 ± 0.60 | 50.17 ± 0.73 |
| Start-conditioned end head | 49.83 ± 0.52 | 50.51 ± 1.02 | 36.84 ± 0.49 | 49.65 ± 0.64 |

Start/end exactness is measured against the deterministic best silver span. It diagnoses boundary selection; it is not a substitute for answer-text metrics.

## 3. Paired qualitative cases

In [4]:
# Compare seed-matched predictions and show the clearest improvements/regressions.
by = {(r["model"], int(r["seed"]), int(r["row_id"])): r for r in details}
pairs = []
ids = sorted({int(r["row_id"]) for r in details})
for seed in (1, 2, 3):
    for rid in ids:
        c = by.get(("BoundaryControl_v2", seed, rid)); n = by.get(("BoundaryAware_start2end", seed, rid))
        if c and n:
            pairs.append((float(n["answer_f1"])-float(c["answer_f1"]), seed, rid, c, n))

def case_table(items, title):
    lines = ["| ΔF1 | Seed | Question | Gold | v2 prediction | New prediction |", "|---:|---:|---|---|---|---|"]
    for d, seed, _, c, n in items:
        clean = lambda x: x.replace("|", "/").replace("\n", " ")
        lines.append(f"| {d:+.1f} | {seed} | {clean(c['question'])} | {clean(c['gold'])} | {clean(c['pred'])} | {clean(n['pred'])} |")
    display(Markdown(f"### {title}\n\n" + "\n".join(lines)))
case_table(sorted(pairs, reverse=True)[:10], "Largest improvements")
case_table(sorted(pairs)[:10], "Largest regressions")


### Largest improvements

| ΔF1 | Seed | Question | Gold | v2 prediction | New prediction |
|---:|---:|---|---|---|---|
| +100.0 | 3 | what was the flight number mentioned? | csn306s | 1 | CSN306S |
| +100.0 | 3 | who did not choose the art or headline? | janet mock | @InterviewMag | Janet Mock |
| +100.0 | 3 | what nation is the number one state of sponsored terror? | iran | numerous violations of Human Rights occurring on an hourly | Iran |
| +100.0 | 3 | who had a career high? | sotnikova. | Bested | Sotnikova |
| +100.0 | 3 | whose face did "she" put her finger? | bb's | Renee Baio | BB'S |
| +100.0 | 3 | who is it that @manutd is hosting? | @lfc | Premier League | @LFC |
| +100.0 | 3 | who established the institution of marriage? | god | woman | God |
| +100.0 | 3 | how many llamas were reportedly on the loose? | 3 | 4 | 3 |
| +100.0 | 3 | trust for public land announced what new land? | ackerson meadow | Yosemite | Ackerson Meadow |
| +100.0 | 3 | how many notre dame players have outscored jabari parker? | six | more points | six |

### Largest regressions

| ΔF1 | Seed | Question | Gold | v2 prediction | New prediction |
|---:|---:|---|---|---|---|
| -100.0 | 1 | who does blake shelton have some of the fondest memories of? | mel tillis | Mel Tillis | most cherished |
| -100.0 | 1 | who was stopped by the cops? | chrisrock | @chrisrock | Tonja Renée Stidhum |
| -100.0 | 1 | what caused the church of san benedetto to collapse? | an earthquake | earthquake | SanBenedetto |
| -100.0 | 1 | who does presiden obama think is an incredible woman. | michelle | Michelle | speech |
| -100.0 | 1 | what is the name of wes welker's horse? | undrafted | Undrafted | Darley #JulyCup |
| -100.0 | 1 | what song was adele singing on the call? | hello | hello | aacps |
| -100.0 | 1 | what has the beam program completed? | a significant milestone | significant milestone | BEAM program |
| -100.0 | 1 | who was texas a&m playing agaisnt? | georgia | Georgia | @AP_Top25Clock stops |
| -100.0 | 1 | who was the opportunity given to? | white women | white women | Kendall Jenner |
| -100.0 | 1 | what kind of firearm is shown in the surveillance stills? | glock pistol | glock pistol | opinion |

In [5]:
report = f"""# Boundary-aware attention experiment

The start-conditioned end head changes mean validation ROUGE-L by {deltas['rougeL']:+.2f},
F1 by {deltas['f1']:+.2f}, and EM by {deltas['em']:+.2f} points relative to the
same TweetBiDAF v2 configuration trained in the same notebook over three seeds.

The boundary table determines whether any change is specifically associated with
start/end selection rather than only an aggregate metric. The example tables show
both improvements and regressions. Results are validation-only; the locked test set
was not accessed. Silver-span exactness is diagnostic because some TweetQA gold
answers are not extractive.
"""
Path("boundary_attention_report.md").write_text(report, encoding="utf-8")
display(Markdown("## 4. Report-ready conclusion\n\n" + report))
print("Saved boundary_attention_report.md")
zf.close()


## 4. Report-ready conclusion

# Boundary-aware attention experiment

The start-conditioned end head changes mean validation ROUGE-L by -0.57,
F1 by -0.60, and EM by -0.56 points relative to the
same TweetBiDAF v2 configuration trained in the same notebook over three seeds.

The boundary table determines whether any change is specifically associated with
start/end selection rather than only an aggregate metric. The example tables show
both improvements and regressions. Results are validation-only; the locked test set
was not accessed. Silver-span exactness is diagnostic because some TweetQA gold
answers are not extractive.


Saved boundary_attention_report.md
